In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 6g pyspark-shell"
)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("als_ml32m_preprocessing")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Master:", spark.sparkContext.master)
print(
    "Driver memory:",
    spark.sparkContext.getConf().get(
        "spark.driver.memory",
        "not configured",
    ),
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/30 20:41:19 WARN Utils: Your hostname, DESKTOP-MU43GAL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/30 20:41:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/anna/projects/ccdpp-pyspark-movielens/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/07/30 20:41:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/30 20:41:23 WARN Util

Master: local[4]
Driver memory: 6g


In [3]:
ratings_32m = (
    spark.read
    .option("header", True)
    .schema(
        "userId INT, "
        "movieId INT, "
        "rating DOUBLE, "
        "timestamp LONG"
    )
    .csv("data/ml-32m/ratings.csv")
    .select(
        F.col("userId").alias("user_id"),
        F.col("movieId").alias("item_id"),
        F.col("rating"),
        F.col("timestamp"),
    )
)

In [4]:
ratings_32m.printSchema()
ratings_32m.show(5)

num_ratings = ratings_32m.count()
print("Ratings:", num_ratings)

assert num_ratings == 32_000_204

root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)



+-------+-------+------+---------+
|user_id|item_id|rating|timestamp|
+-------+-------+------+---------+
|      1|     17|   4.0|944249077|
|      1|     25|   1.0|944250228|
|      1|     29|   2.0|943230976|
|      1|     30|   5.0|944249077|
|      1|     32|   5.0|943228858|
+-------+-------+------+---------+
only showing top 5 rows


Ratings: 32000204


In [5]:
ratings_32m.agg(
    F.countDistinct("user_id").alias("users"),
    F.countDistinct("item_id").alias("items"),
    F.min("rating").alias("min_rating"),
    F.max("rating").alias("max_rating"),
).show()

+------+-----+----------+----------+
| users|items|min_rating|max_rating|
+------+-----+----------+----------+
|200948|84432|       0.5|       5.0|
+------+-----+----------+----------+



split

In [6]:
ratings_with_bucket = ratings_32m.withColumn(
    "split_bucket",
    F.pmod(
        F.xxhash64(
            "user_id",
            "item_id",
            "timestamp",
        ),
        F.lit(10),
    ),
)

In [7]:
train_32m = (
    ratings_with_bucket
    .filter(F.col("split_bucket") != 9)
    .drop("split_bucket")
)

test_32m = (
    ratings_with_bucket
    .filter(F.col("split_bucket") == 9)
    .drop("split_bucket")
)

In [8]:
from pyspark import StorageLevel

train_32m.persist(StorageLevel.MEMORY_AND_DISK)
test_32m.persist(StorageLevel.MEMORY_AND_DISK)

train_count = train_32m.count()
test_count = test_32m.count()

print("Train:", train_count)
print("Test:", test_count)
print("Total:", train_count + test_count)

assert train_count + test_count == 32_000_204

Train: 28801756
Test: 3198448
Total: 32000204


In [9]:
train_users = train_32m.select("user_id").distinct()
train_items = train_32m.select("item_id").distinct()

test_users = test_32m.select("user_id").distinct()
test_items = test_32m.select("item_id").distinct()

In [10]:
unseen_users = test_users.join(
    train_users,
    "user_id",
    "left_anti",
)

unseen_items = test_items.join(
    train_items,
    "item_id",
    "left_anti",
)

print("Test-only users:", unseen_users.count())
print("Test-only items:", unseen_items.count())

Test-only users: 0


Test-only items: 1908


In [11]:
test_clean_32m = (
    test_32m
    .join(train_users, "user_id", "left_semi")
    .join(train_items, "item_id", "left_semi")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

test_clean_count = test_clean_32m.count()
coverage = test_clean_count / test_count

print("Original test rows:", test_count)
print("Clean test rows:", test_clean_count)
print("Coverage:", coverage)

Original test rows: 3198448
Clean test rows: 3196413
Coverage: 0.9993637539206515


In [12]:
train_32m.write.mode("overwrite").parquet(
    "data/processed/ml-32m/train"
)

test_clean_32m.write.mode("overwrite").parquet(
    "data/processed/ml-32m/test_clean"
)

In [13]:
train_32m.unpersist()
test_32m.unpersist()
test_clean_32m.unpersist()

DataFrame[item_id: int, user_id: int, rating: double, timestamp: bigint]

In [14]:
train = (
    spark.read
    .parquet("data/processed/ml-32m/train")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

test = (
    spark.read
    .parquet("data/processed/ml-32m/test_clean")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

train_count = train.count()
test_count = test.count()

print("Train rows:", train_count)
print("Test rows:", test_count)

Train rows: 28801756
Test rows: 3196413


In [15]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from time import perf_counter

als = ALS(
    userCol="user_id",
    itemCol="item_id",
    ratingCol="rating",
    rank=5,
    regParam=0.1,
    maxIter=10,
    numUserBlocks=4,
    numItemBlocks=4,
    coldStartStrategy="drop",
    seed=42,
)

evaluator = RegressionEvaluator(
    labelCol="rating",
    predictionCol="prediction",
    metricName="rmse",
)


_ = als.fit(train)


start = perf_counter()
model = als.fit(train)
training_time = perf_counter() - start

In [16]:
predictions = model.transform(test).persist(
    StorageLevel.MEMORY_AND_DISK
)

prediction_count = predictions.count()
rmse = evaluator.evaluate(predictions)
prediction_coverage = prediction_count / test_count

print("Training time:", training_time)
print("RMSE:", rmse)
print("Prediction rows:", prediction_count)
print("Test rows:", test_count)
print("Coverage:", prediction_coverage)

predictions.unpersist()

Training time: 160.99892714500038
RMSE: 0.8132186189489738
Prediction rows: 3196413
Test rows: 3196413
Coverage: 1.0


DataFrame[item_id: int, user_id: int, rating: double, timestamp: bigint, prediction: float]

In [17]:
spark.stop()